<a href="https://colab.research.google.com/github/Sanad-Manga/NYC-Build-And-Crash/blob/main/01_housing_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

sample_url = "https://data.cityofnewyork.us/resource/w9ak-ipjd.csv?$limit=1000"
df_sample = pd.read_csv(sample_url, dtype=str)

print(df_sample.shape)
print(df_sample.columns.tolist())

(1000, 95)
['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board', 'work_on_floor', 'apt_condo_no_s', 'applicant_professional_title', 'applicant_license', 'applicant_first_name', 'applicants_middle_initial', 'applicant_last_name', 'owner_s_business_name', 'applicant_street_name', 'city', 'state', 'zip', 'filing_representative_first_name', 'filing_representative_middle_initial', 'filing_representative_last_name', 'filing_representative_business_name', 'filing_representative_street_name', 'filing_representative_city', 'filing_representative_state', 'filing_representative_zip', 'sprinkler_work_type', 'plumbing_work_type', 'standpipe', 'antenna', 'sign', 'curb_cut', 'fence', 'scaffold', 'shed', 'initial_cost', 'total_construction_floor_area', 'review_building_code', 'little_e', 'unmapped_cco_street', 'request_legalization', 'includes_permanent_removal', 'in_compliance_with_nycecc', 'exempt_from_nycecc', 'building_type', 'exist

In [3]:
for column in df_sample.columns:
    print(column)

count_url = "https://data.cityofnewyork.us/resource/w9ak-ipjd.csv?$select=count(*)"
print(pd.read_csv(count_url))

job_filing_number
filing_status
house_no
street_name
borough
block
lot
bin
commmunity_board
work_on_floor
apt_condo_no_s
applicant_professional_title
applicant_license
applicant_first_name
applicants_middle_initial
applicant_last_name
owner_s_business_name
applicant_street_name
city
state
zip
filing_representative_first_name
filing_representative_middle_initial
filing_representative_last_name
filing_representative_business_name
filing_representative_street_name
filing_representative_city
filing_representative_state
filing_representative_zip
sprinkler_work_type
plumbing_work_type
standpipe
antenna
sign
curb_cut
fence
scaffold
shed
initial_cost
total_construction_floor_area
review_building_code
little_e
unmapped_cco_street
request_legalization
includes_permanent_removal
in_compliance_with_nycecc
exempt_from_nycecc
building_type
existing_stories
existing_height
existing_dwelling_units
proposed_no_of_stories
proposed_height
proposed_dwelling_units
specialinspectionrequirement
special_inspe

In [4]:
base_url = "https://data.cityofnewyork.us/resource/w9ak-ipjd.csv"
batch_size = 50000
offset = 0
batches = []

while True:
    url = f"{base_url}?$limit={batch_size}&$offset={offset}&$order=:id"
    batch = pd.read_csv(url, dtype=str)
    batches.append(batch)
    offset += len(batch)
    print("Rows downloaded so far:", offset)

    if len(batch) < batch_size:
        break

df_housing = pd.concat(batches, ignore_index=True)
df_housing.to_csv("housing_raw.csv", index=False)

print("Finished!", df_housing.shape)

Rows downloaded so far: 50000
Rows downloaded so far: 100000
Rows downloaded so far: 150000
Rows downloaded so far: 200000
Rows downloaded so far: 250000
Rows downloaded so far: 300000
Rows downloaded so far: 350000
Rows downloaded so far: 400000
Rows downloaded so far: 450000
Rows downloaded so far: 500000
Rows downloaded so far: 550000
Rows downloaded so far: 600000
Rows downloaded so far: 650000
Rows downloaded so far: 700000
Rows downloaded so far: 750000
Rows downloaded so far: 800000
Rows downloaded so far: 850000
Rows downloaded so far: 900000
Rows downloaded so far: 950000
Rows downloaded so far: 977813
Finished! (977813, 95)


In [5]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")
shutil.copy("housing_raw.csv", "/content/drive/MyDrive/housing_raw.csv")

print("Saved to Drive!")
print(round(os.path.getsize("housing_raw.csv") / 1024**3, 2), "GB")

Mounted at /content/drive
Saved to Drive!
0.79 GB


In [6]:
import gc

del batches
gc.collect()

missing_pct = df_housing.isna().mean().sort_values(ascending=False) * 100
print(df_housing.shape)
print(missing_pct.round(1).to_string())

(977813, 95)
request_legalization                        100.0
includes_permanent_removal                  100.0
apt_condo_no_s                               98.1
filing_representative_middle_initial         93.1
built_2_b_information_value                  89.9
built_2_a_information_value                  89.9
built_2_information_value                    89.9
built_1_information_value                    89.9
applicants_middle_initial                    69.8
existing_height                              69.6
proposed_height                              65.0
existing_stories                             61.1
signoff_date                                 60.1
proposed_no_of_stories                       55.9
special_inspection_agency_number             43.5
progressinspectionrequirement                40.2
first_permit_date                            39.6
specialinspectionrequirement                 37.4
unmapped_cco_street                          35.8
filing_representative_street_name    

## Handling missing values

**Columns removed:**
- `request_legalization` and `includes_permanent_removal` are 100% empty.
- Applicant, owner, and filing representative details don't help answer our question, and they include personal information.
- The `built_*` columns are about 90% empty. `existing_stories` and `proposed_no_of_stories` are 56–61% empty, and `total_construction_floor_area` is a more complete measure of size.
- Other permit and administrative columns aren't needed for our research questions.

**Columns kept despite missing values:**
- `first_permit_date` and `approved_date` may be empty because a job hasn't reached that stage yet. We need to verify this before relying on it.
- `existing_dwelling_units`, `proposed_dwelling_units`, and `total_construction_floor_area` stay as missing. Filling them with an average would invent data.

**Rows removed:** rows missing `job_filing_number`, `filing_date`, or `postcode`, because they can't be identified or placed in a ZIP code and year. This is a very small share of the data.

In [7]:
keep_columns = [
    "job_filing_number", "filing_status", "job_type", "building_type",
    "filing_date", "approved_date", "first_permit_date", "current_status_date",
    "borough", "postcode", "latitude", "longitude", "nta", "census_tract",
    "council_district", "commmunity_board", "bbl",
    "initial_cost", "total_construction_floor_area",
    "existing_dwelling_units", "proposed_dwelling_units",
    "full_demolition_work_type_", "job_description",
]

df_clean = df_housing[keep_columns].copy()

# Trim spaces and treat blank text as missing
df_clean = df_clean.apply(lambda column: column.str.strip()).replace("", pd.NA)

# Remove rows we can't identify or place in a ZIP code and year
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=["job_filing_number", "filing_date", "postcode"])
rows_removed = rows_before - len(df_clean)

print("Rows removed for missing values:", rows_removed)
print("Percent removed:", round(rows_removed / rows_before * 100, 2))
print("Rows left:", len(df_clean))

Rows removed for missing values: 4985
Percent removed: 0.51
Rows left: 972828


In [8]:
exact_duplicates = df_clean.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)
df_clean = df_clean.drop_duplicates()

print("Repeated filing numbers:", df_clean["job_filing_number"].duplicated().sum())
print(df_clean["job_filing_number"].head(10).to_string())

filing_suffix = df_clean["job_filing_number"].str.extract(r"-([^-]+)$")[0]
print(filing_suffix.value_counts(dropna=False).head(15).to_string())
print(df_clean["job_type"].value_counts(dropna=False).to_string())

Exact duplicate rows: 28
Repeated filing numbers: 121
0    B00319790-I1
1    B00422847-I1
2    B00455182-I1
3    B00244768-P3
4    B00464221-S2
5    B00451661-P2
6    B00478927-I1
7    B00004392-I1
8    B00206681-P2
9    B00388013-I1
0
I1    569326
P1    147912
S1     60380
P2     57305
S2     27921
P3     26637
S3     15624
P4     13960
S4      8864
P5      8081
S5      5793
P6      5136
S6      4075
P7      3423
S7      2779
job_type
Alteration                                                820220
New Building                                               57757
Alteration CO                                              51954
No Work                                                    18189
ALT-CO - New Building with Existing Elements to Remain     16691
Full Demolition                                             7989


In [9]:
print("Sample dates:", df_clean["current_status_date"].head(3).tolist())

# Temporary date column, used only to find the newest version of each filing
df_clean["status_date_temp"] = pd.to_datetime(df_clean["current_status_date"], format="mixed", errors="coerce")
print("Dates that couldn't be read:", df_clean["status_date_temp"].isna().sum())

# See which columns differ between the repeated rows
repeated = df_clean[df_clean["job_filing_number"].duplicated(keep=False)]
differences = repeated.groupby("job_filing_number").nunique(dropna=False) > 1
print(differences.sum().sort_values(ascending=False).head(8).to_string())

# Keep the newest version of each filing
df_clean = (df_clean.sort_values("status_date_temp")
                    .drop_duplicates("job_filing_number", keep="last")
                    .drop(columns="status_date_temp"))

print("Repeated filing numbers now:", df_clean["job_filing_number"].duplicated().sum())
print("Rows left:", len(df_clean))

Sample dates: ['2021-07-26T15:41:01.000', '2020-12-27T20:31:03.000', '2021-01-19T20:27:46.000']
Dates that couldn't be read: 1
filing_status                    70
approved_date                    31
filing_date                      31
current_status_date              31
status_date_temp                 31
initial_cost                     24
job_description                  23
total_construction_floor_area     9
Repeated filing numbers now: 0
Rows left: 972679


In [10]:
df_clean["job_id"] = df_clean["job_filing_number"].str.split("-").str[0]
df_clean["is_initial_filing"] = df_clean["job_filing_number"].str.endswith("-I1")
df_clean = df_clean.rename(columns={"postcode": "zip_code"})

jobs_total = df_clean["job_id"].nunique()
jobs_with_i1 = df_clean.loc[df_clean["is_initial_filing"], "job_id"].nunique()

print("Unique jobs:", jobs_total)
print("Jobs with an I1 filing:", jobs_with_i1)
print("Jobs without an I1 filing:", jobs_total - jobs_with_i1)
print("Job IDs with more than one job type:", (df_clean.groupby("job_id")["job_type"].nunique() > 1).sum())

Unique jobs: 569696
Jobs with an I1 filing: 569326
Jobs without an I1 filing: 370
Job IDs with more than one job type: 15


In [11]:
from google.colab import drive
import os

drive.mount("/content/drive")

output_path = "/content/drive/MyDrive/housing_step1.csv.gz"
df_clean.to_csv(output_path, index=False)

print("Saved!", df_clean.shape)
print(round(os.path.getsize(output_path) / 1024**2), "MB")
print("Raw backup in Drive:", os.path.exists("/content/drive/MyDrive/housing_raw.csv"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved! (972679, 25)
87 MB
Raw backup in Drive: True


## Handling duplicates

| Step | Rows removed | Rows left |
|---|---:|---:|
| Starting data | – | 977,813 |
| Missing filing number, date, or ZIP | 4,985 | 972,828 |
| Exact duplicate rows | 28 | 972,800 |
| Older versions of repeated filing numbers | 121 | 972,679 |

**Exact duplicates** were removed because they add no information.

**Repeated filing numbers:** these rows mostly differ in status, dates, or cost, which suggests updates to the same filing. We kept the row with the latest `current_status_date`.

**Multiple filings per job:** the 972,679 rows represent 569,696 jobs, because one job can have several filings. We kept these rows and added:
- `job_id`: the job number without its filing suffix
- `is_initial_filing`: true for `I1` filings

Counting `I1` rows gives one row per job and avoids counting a job more than once. This misses 370 jobs with no `I1` row (0.06%). Also, 15 job IDs have more than one job type, so we use the type on the `I1` row.

**Contribution:** Mohamed Issa loaded the housing data, handled missing values and duplicates, and prepared it for the next cleaning stage.